# 《深山传家》Colab 批量 2.5D 烘培 + 明清木刻工笔风格化

用法：
1. 云盘 `shenshan_td/raw_models/` 里放好 `.glb`（由 `colab_gen_3d.ipynb` 生成，文件名 = `assets_manifest.json` 的 `id`）。
2. 菜单「代码执行程序 → 更改运行时类型」选 **GPU（T4 / L4 / A100 均可，无需 High-RAM）**。
3. `SYNC_FROM_GITHUB=True` 会先从公开仓库拉最新的 manifest、烘培脚本、风格化脚本与 `style_profile.json` 到云盘。
4. 先把 `TEST_SINGLE` 设为单个文件名试看：输出到 `baked_sprites/_preview/` 并强制覆盖，不会挡住之后的正式批量烘培；满意后留空，`SCALE=1.0, SAMPLES=48` 正式烘培全部模型到 `baked_sprites/`。
5. 只改了 `style_profile.json` 想重出图、不想重新渲染：设 `RESTYLE_ONLY=True`（几秒一件）。
6. 最后一格自动释放运行时。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/shenshan_td"
SYNC_FROM_GITHUB = True    # 先从公开仓库同步最新脚本与画风参数到云盘
GITHUB_RAW = "https://raw.githubusercontent.com/dongdongw12/shenshan_td_project/main"
BLENDER = "5.3.0"          # 与本机 Blender 版本一致（脚本也兼容 5.0.x）
TEST_SINGLE = "house_lv2_tile.glb"  # 填文件名只试烘 1 个（输出到 _preview/）；留空 "" 正式烘培全部
MODE = "stylized"          # "stylized" = 工笔风格化（默认）| "classic" = v1 直出（仅对比用）
SCALE = 0.5                # stylized：比例倍率，0.5 预览 / 1.0 正式（128 px/m）
SAMPLES = 24               # Cycles 采样数（正式批量建议 48）
DEVICE = "auto"            # OPTIX 报错则改为 "CUDA"
OVERWRITE = False          # True 会重烘已有文件（TEST_SINGLE 模式总是覆盖）
RESTYLE_ONLY = False       # True = 不渲染，只用已有 _passes 重新风格化

In [ ]:
import os, subprocess, urllib.request
from google.colab import drive
drive.mount('/content/drive')

SYNC = {"assets_manifest.json": "assets_manifest.json",
        "pipeline/blender/bake_isometric_sprites.py": "bake_isometric_sprites.py",
        "pipeline/style/stylize_sprites.py": "stylize_sprites.py",
        "pipeline/style/style_profile.json": "style_profile.json"}
if SYNC_FROM_GITHUB:
    for src, dst in SYNC.items():
        urllib.request.urlretrieve(f"{GITHUB_RAW}/{src}", f"{DRIVE_DIR}/{dst}")
    print("已从 GitHub 同步：", ", ".join(SYNC.values()))
for dst in SYNC.values():
    assert os.path.exists(f"{DRIVE_DIR}/{dst}"), f"云盘缺少 {dst}"
assert os.path.exists(f"{DRIVE_DIR}/raw_models"), "请先建立 raw_models 目录并放入 .glb 模型"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

mm = ".".join(BLENDER.split(".")[:2])
if not os.path.exists("/content/blender/blender"):
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 libegl1 > /dev/null
    !wget -q https://download.blender.org/release/Blender{mm}/blender-{BLENDER}-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender --version | head -1
has_oiio = "OK" in subprocess.run(["/content/blender/blender", "-b", "--python-expr", "import OpenImageIO; print('OK')"],
                                  capture_output=True, text=True).stdout
if not has_oiio:
    !pip install -q OpenEXR pillow
print("风格化在 Blender 内运行" if has_oiio else "风格化改用 Colab Python 运行")

In [ ]:
in_target = f"{DRIVE_DIR}/raw_models/{TEST_SINGLE}" if TEST_SINGLE else f"{DRIVE_DIR}/raw_models"
out_dir = f"{DRIVE_DIR}/baked_sprites/_preview" if TEST_SINGLE else f"{DRIVE_DIR}/baked_sprites"
profile = f"{DRIVE_DIR}/style_profile.json"
only = os.path.splitext(TEST_SINGLE)[0] if TEST_SINGLE else ""

if not RESTYLE_ONLY:
    opts = f"--mode {MODE} --scale {SCALE} --samples {SAMPLES} --device {DEVICE} --profile '{profile}'"
    opts += " --overwrite" if (OVERWRITE or TEST_SINGLE) else ""
    opts += "" if (has_oiio or MODE == "classic") else " --skip-stylize"
    !/content/blender/blender -b -P "{DRIVE_DIR}/bake_isometric_sprites.py" -- --input "{in_target}" --out "{out_dir}" --manifest "{DRIVE_DIR}/assets_manifest.json" {opts} 2>&1 | grep -E "^BAKE|^START|^RENDERED|^BAKED|^SKIP|^ALL_|Error|Traceback"
if MODE == "stylized" and (RESTYLE_ONLY or not has_oiio):
    py = f"/content/blender/{mm}/python/bin/python3*" if has_oiio else "python"
    !{py} "{DRIVE_DIR}/stylize_sprites.py" --passes "{out_dir}/_passes" --out "{out_dir}" --profile "{profile}" --only "{only}"

In [ ]:
# 烘培完毕自动释放 Colab GPU 运行时，防止空转扣费
from google.colab import runtime
runtime.unassign()